In [22]:
# load env variables & create client
from dotenv import load_dotenv

load_dotenv()

# create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

def add_user_message(messages, text):
    user_message = { "role": "user", "content": text}
    messages.append(user_message)
def add_assistant_message(messages, text):
    assistant_message = { "role": "assistant", "content": text}
    messages.append(assistant_message)


#when using streaming, the chat function works a bit differently, so we're now manually calling 
# the client.messages.create function in the next part of the notebook
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model":model,
        "max_tokens":1000,
        "messages": messages,
        "temperature": temperature
    }

#This approach handles an important detail: Claude's API doesn't accept system=None, 
# so you need to conditionally include the system parameter only when it's provided.
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message.content[0].text

In [23]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json. Output it between <json> and </json> as am I practicing stop_sequences. Output ONLY the JSON")
#This is different compared to the course, as the explanation in the course is no longer valid (you cannot end with an assistant message in Sonnet anymore). This solution was proposed by Claude itself
text = chat(messages, stop_sequences=["</json>"])
text

'<json>\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n'

In [24]:
import json
# the first .strip() removes any leading/trailing whitespace from the full response, then .removeprefix("<json>") strips the tag, and the second .strip() removes any whitespace/newline that was sitting right after the tag. 
json.loads(text.strip().removeprefix("<json>").strip())

{'source': ['aws.ec2'],
 'detail-type': ['EC2 Instance State-change Notification'],
 'detail': {'state': ['running']}}

In [25]:
#Exercise

messages = []

prompt = """
Generate three diferent sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
#since you're not allowed to end with add_assistant_message anymore, had to get a bit creative with adding a system prompt
text = chat(
    messages,
    system="Output only a numbered list. No introduction, no explanation, no trailing text. Start immediately with '1.'",
    stop_sequences=["4."]
)
text.strip();


In [ ]:
from IPython.display import Markdown

Markdown(text)

1. `aws s3 ls`
2. `aws ec2 describe-instances`
3. `aws iam list-users`

: 